# InterpretML for Explainability in Machine Learning
### *Chapter 3 — XAI Techniques | Explainable AI in Medical Systems*

---

**InterpretML** is an open-source framework from Microsoft that uniquely unifies **glassbox**
(intrinsically interpretable) and **blackbox** (post-hoc) explanation methods under a single API.
It is the only major XAI library that implements the **Explainable Boosting Machine (EBM)** —
the most powerful intrinsically interpretable model in widespread clinical use — alongside
wrappers for LIME, SHAP, Partial Dependence Plots, and Morris Sensitivity Analysis.

### Two explanation philosophies in one library

| Category | Models | Explanation type |
|---|---|---|
| **Glassbox** | EBM, Logistic Regression, Decision Tree | Intrinsic — model IS the explanation |
| **Blackbox** | LIME, ShapKernel, PDP, Morris Sensitivity | Post-hoc — applied to any model |

### Why InterpretML matters for medical XAI

Chapter 4 shows that 78% of medical XAI papers rely exclusively on post-hoc methods applied
to black-box models, while intrinsically interpretable models appear in fewer than 2% of papers.
InterpretML makes the EBM practically accessible: a model that is *simultaneously*
high-performing and transparent, with no post-hoc approximation needed.

---

## Contents

1. [Setup and dataset](#1)
2. [Glassbox: Explainable Boosting Machine (EBM)](#2)
   - 2.1 Training and performance
   - 2.2 Global feature importance
   - 2.3 Shape functions — non-linear feature effects
   - 2.4 Local explanation — individual patient
3. [Glassbox: Logistic Regression](#3)
4. [Glassbox: Decision Tree](#4)
5. [Glassbox comparison — EBM vs LR vs Tree](#5)
6. [Blackbox: LIME on Random Forest](#6)
7. [Blackbox: ShapKernel on Random Forest](#7)
8. [Blackbox: Partial Dependence Plot](#8)
9. [Blackbox: Morris Sensitivity Analysis](#9)
10. [Performance comparison](#10)
11. [Summary](#11)


<a id='1'></a>
## 1. Setup and dataset


In [ ]:
# Install required packages (run once)
# !pip install interpret scikit-learn pandas numpy matplotlib lime shap


In [ ]:
import interpret
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings

from interpret.glassbox import (
    ExplainableBoostingClassifier,
    LogisticRegression,
    ClassificationTree
)
from interpret.blackbox import (
    LimeTabular,
    ShapKernel,
    PartialDependence,
    MorrisSensitivity
)

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

warnings.filterwarnings('ignore')
print(f'InterpretML version : {interpret.__version__}')


In [ ]:
# ── Wisconsin Breast Cancer Dataset ──────────────────────────────────────────
data = load_breast_cancer()
X    = pd.DataFrame(data.data, columns=data.feature_names)
y    = pd.Series(data.target, name='diagnosis')  # 0=malignant, 1=benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardised features for Logistic Regression
scaler     = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

feat_names = list(X_train.columns)

print(f'Training set : {X_train.shape[0]} samples  |  Test set : {X_test.shape[0]} samples')
print(f'Features     : {X_train.shape[1]}')
print(f'Classes      : malignant (0), benign (1)')


<a id='2'></a>
## 2. Glassbox: Explainable Boosting Machine (EBM)

The **Explainable Boosting Machine** is a Generalised Additive Model (GAM) with pairwise
interactions, trained using gradient boosting on each feature in round-robin order.
Its additive structure makes it intrinsically interpretable:

$$f(x) = \beta_0 + \sum_i f_i(x_i) + \sum_{i<j} f_{ij}(x_i, x_j)$$

Each term $f_i(x_i)$ is a **shape function** — a learned, piecewise-linear mapping of feature
$x_i$ to its log-odds contribution — computed exactly, not approximated post-hoc.

**Key properties for clinical AI:**
- Performance competitive with Random Forest and XGBoost on tabular data
- Faithfulness guaranteed by construction — the shape function IS the model
- Confidence intervals on shape functions quantify explanation uncertainty
- Directly satisfies EU AI Act Article 13 transparency requirements


In [ ]:
# ── 2.1 Train EBM ────────────────────────────────────────────────────────────
ebm = ExplainableBoostingClassifier(
    random_state  = 42,
    n_jobs        = 1,
    learning_rate = 0.01,
    max_rounds    = 5000
)
ebm.fit(X_train, y_train)

auc_ebm = roc_auc_score(y_test, ebm.predict_proba(X_test)[:, 1])
print(f'EBM AUC-ROC : {auc_ebm:.4f}')
print(classification_report(y_test, ebm.predict(X_test),
                             target_names=['malignant', 'benign']))


### 2.2 Global feature importance

The global explanation shows the mean absolute contribution of each feature across all
training samples. Unlike SHAP global importance — which aggregates local post-hoc attributions
— the EBM global importance is derived directly from the learned shape functions and is exact.


In [ ]:
# ── 2.2 EBM Global Explanation ───────────────────────────────────────────────
ebm_global  = ebm.explain_global(name='EBM Global')
global_data = ebm_global.data()
gnames      = global_data['names']
gscores     = global_data['scores']
sorted_idx  = np.argsort(gscores)

fig, ax = plt.subplots(figsize=(9, 6))
top_n   = 15
colours = plt.cm.Blues(np.linspace(0.4, 0.9, top_n))
ax.barh(range(top_n), [gscores[i] for i in sorted_idx[-top_n:]], color=colours)
ax.set_yticks(range(top_n))
ax.set_yticklabels([gnames[i] for i in sorted_idx[-top_n:]], fontsize=9)
ax.set_xlabel('Mean absolute contribution (global importance)', fontsize=10)
ax.set_title(
    'EBM Global Feature Importance\n'
    'Intrinsic — derived directly from learned shape functions, not post-hoc approximation',
    fontsize=10, pad=10
)
plt.tight_layout()
plt.show()

print('Top 5 features:')
for i in sorted_idx[-5:][::-1]:
    print(f'  {gnames[i]:<35s}  {gscores[i]:.4f}')


### 2.3 Shape functions — non-linear feature effects

The shape function shows exactly how the EBM prediction changes as a feature varies across
its observed range. It is equivalent to a Partial Dependence Plot but **exact** rather than
approximated — computed analytically from the model's internal representation.
The shaded confidence interval shows where the model is uncertain about the shape,
derived from the variance across the ensemble of boosted trees.


In [ ]:
# ── 2.3 EBM Shape Functions (top 4 features) ─────────────────────────────────
top4_idx = [int(i) for i in sorted_idx[-4:]]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax_i, feat_i in enumerate(reversed(top4_idx)):
    fd = ebm.explain_global().data(feat_i)
    # EBM stores bin edges (length n+1) and bin scores (length n)
    edges  = np.array(fd['names'])   # n+1 bin edges
    y_vals = np.array(fd['scores'])  # n bin values
    ub     = fd.get('upper_bounds', None)
    lb     = fd.get('lower_bounds', None)
    x_ctr  = (edges[:-1] + edges[1:]) / 2   # bin centres

    axes[ax_i].plot(x_ctr, y_vals, color='#2E75B6', linewidth=2, zorder=3)
    if ub is not None and lb is not None:
        axes[ax_i].fill_between(x_ctr, lb, ub, alpha=0.2,
                                 color='#2E75B6', label='95% CI', zorder=2)
    axes[ax_i].axhline(0, color='black', linewidth=0.8,
                        linestyle='--', alpha=0.5, label='Baseline')
    axes[ax_i].set_xlabel(gnames[feat_i], fontsize=9)
    axes[ax_i].set_ylabel('Shape value (log-odds)', fontsize=8)
    axes[ax_i].set_title(f'Shape: {gnames[feat_i]}', fontsize=9)
    axes[ax_i].legend(fontsize=7)
    axes[ax_i].spines['top'].set_visible(False)
    axes[ax_i].spines['right'].set_visible(False)

fig.suptitle(
    'EBM Shape Functions — Top 4 Features\n'
    'Positive values push toward benign | Shaded region = 95% CI',
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.show()


### 2.4 Local explanation — individual patient

The local explanation shows the **exact contribution** of each feature to a single patient's
prediction, decomposing the log-odds output into a sum of per-feature contributions plus
an intercept. Because the EBM is additive, these contributions are exact — no sampling,
no surrogate fitting. This is the most important advantage over LIME and SHAP KernelExplainer.


In [ ]:
# ── 2.4 EBM Local Explanation ─────────────────────────────────────────────────
patient_idx = 3
true_label  = 'benign' if y_test.iloc[patient_idx] == 1 else 'malignant'
pred_prob   = ebm.predict_proba(X_test.iloc[[patient_idx]])[0, 1]
pred_label  = 'benign' if pred_prob > 0.5 else 'malignant'

print(f'Patient {patient_idx}')
print(f'  True label      : {true_label}')
print(f'  Predicted       : {pred_label}  (P(benign) = {pred_prob:.4f})')

ebm_local  = ebm.explain_local(
    X_test.iloc[patient_idx:patient_idx+1],
    y_test.iloc[patient_idx:patient_idx+1]
)
local_data = ebm_local.data(0)
names  = local_data['names']
scores = local_data['scores']
values = local_data['values']

top_idx = np.argsort(np.abs(scores))[-12:]
t_names   = [names[i]  for i in top_idx]
t_scores  = [scores[i] for i in top_idx]
t_values  = [values[i] for i in top_idx]
colours   = ['#2ecc71' if s > 0 else '#e74c3c' for s in t_scores]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(t_names)), t_scores, color=colours)
ax.set_yticks(range(len(t_names)))
ax.set_yticklabels(
    [f'{n} = {v:.3f}' for n, v in zip(t_names, t_values)],
    fontsize=8.5
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('EBM contribution (exact, log-odds scale)', fontsize=10)
ax.set_title(
    f'EBM Local Explanation — Patient {patient_idx}\n'
    f'True: {true_label} | Predicted: {pred_label} | P(benign) = {pred_prob:.3f}',
    fontsize=10, pad=10
)
g_p = mpatches.Patch(color='#2ecc71', label='Pushes toward benign')
r_p = mpatches.Patch(color='#e74c3c', label='Pushes toward malignant')
ax.legend(handles=[g_p, r_p], fontsize=8.5)
plt.tight_layout()
plt.show()

print('EBM local contributions are exact — computed analytically, not via sampling.')


<a id='3'></a>
## 3. Glassbox: Logistic Regression

InterpretML wraps sklearn's Logistic Regression with native explanation support.
The global explanation shows standardised coefficients — directly interpretable as the
change in log-odds per one-standard-deviation change in each feature.
This is the most auditable model for regulatory submissions.


In [ ]:
# ── 3. Logistic Regression ────────────────────────────────────────────────────
lr = LogisticRegression(random_state=42, n_jobs=1)
lr.fit(X_train_sc, y_train)
auc_lr = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:, 1])
print(f'Logistic Regression AUC-ROC : {auc_lr:.4f}')

lr_global = lr.explain_global(name='LR Global')
lr_data   = lr_global.data()
lr_names  = lr_data['names']
lr_scores = lr_data['scores']
sorted_lr = np.argsort(np.abs(lr_scores))[-15:]
bar_cols  = ['#2ecc71' if lr_scores[i] > 0 else '#e74c3c' for i in sorted_lr]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(len(sorted_lr)), [lr_scores[i] for i in sorted_lr], color=bar_cols)
ax.set_yticks(range(len(sorted_lr)))
ax.set_yticklabels([lr_names[i] for i in sorted_lr], fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (standardised features)', fontsize=10)
ax.set_title('InterpretML Logistic Regression — Global Coefficients', fontsize=10, pad=10)
g_p = mpatches.Patch(color='#2ecc71', label='Positive: pushes toward benign')
r_p = mpatches.Patch(color='#e74c3c', label='Negative: pushes toward malignant')
ax.legend(handles=[g_p, r_p], fontsize=8.5)
plt.tight_layout()
plt.show()


<a id='4'></a>
## 4. Glassbox: Decision Tree

`ClassificationTree` wraps sklearn's `DecisionTreeClassifier` with InterpretML's explanation
API. Feature importance is Gini-based. The decision path for a specific patient traces each
splitting rule applied to arrive at the predicted class — the most directly auditable
explanation format for clinical rule-based reasoning.


In [ ]:
# ── 4. Classification Tree ────────────────────────────────────────────────────
tree = ClassificationTree(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
auc_tree = roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1])
print(f'Decision Tree (depth=4) AUC-ROC : {auc_tree:.4f}')

# Access underlying sklearn model for feature importance and decision path
sk_tree     = tree.sk_model_
imps        = sk_tree.feature_importances_
sorted_tree = np.argsort(imps)[-12:]

fig, ax = plt.subplots(figsize=(9, 5))
colours = plt.cm.Blues(np.linspace(0.4, 0.9, 12))
ax.barh(range(12), [imps[i] for i in sorted_tree], color=colours)
ax.set_yticks(range(12))
ax.set_yticklabels([feat_names[i] for i in sorted_tree], fontsize=9)
ax.set_xlabel('Gini-based feature importance', fontsize=10)
ax.set_title('InterpretML Decision Tree — Global Feature Importance', fontsize=10, pad=10)
plt.tight_layout()
plt.show()

# Decision path for patient_idx
dp          = sk_tree.decision_path(X_test.iloc[[patient_idx]])
node_ids    = dp.indices
feat_idx_dp = sk_tree.tree_.feature
thresholds  = sk_tree.tree_.threshold
x_vals_pt   = X_test.iloc[patient_idx]

print(f'Decision path for Patient {patient_idx} (True: {true_label}):')
for node in node_ids[:-1]:  # exclude the leaf node
    fi        = feat_idx_dp[node]
    thr       = thresholds[node]
    val       = x_vals_pt.iloc[fi]
    direction = '<= (left)' if val <= thr else '> (right)'
    print(f'  {feat_names[fi]:<32s} = {val:.3f}  vs {thr:.3f}  -> {direction}')


<a id='5'></a>
## 5. Glassbox comparison — EBM vs Logistic Regression vs Decision Tree

All three glassbox models agree on the most important features but represent the
feature-outcome relationship differently. LR assumes linearity; the Decision Tree
uses axis-aligned splits; the EBM learns arbitrary non-linear shapes. The comparison
reveals where linearity assumptions may be inadequate for a given clinical variable.


In [ ]:
# ── 5. Glassbox comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

panels = [
    ('EBM',                gnames,     gscores,              plt.cm.Blues),
    ('Logistic Regression', lr_names,  [abs(s) for s in lr_scores], plt.cm.Greens),
    ('Decision Tree',       feat_names, imps,                 plt.cm.Oranges),
]

for ax, (mname, mnames, mscores, cmap) in zip(axes, panels):
    s_idx = np.argsort(mscores)[-10:]
    colours = cmap(np.linspace(0.4, 0.9, 10))
    ax.barh(range(10), [mscores[i] for i in s_idx], color=colours)
    ax.set_yticks(range(10))
    ax.set_yticklabels([mnames[i][:28] for i in s_idx], fontsize=7.5)
    ax.set_title(mname, fontsize=9.5, fontweight='bold')
    ax.set_xlabel('Importance score', fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Glassbox Models: Feature Importance Rankings Compared',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Glassbox performance summary:')
print(f'  EBM (non-linear GAM)     : AUC = {auc_ebm:.4f}')
print(f'  Logistic Regression (LR) : AUC = {auc_lr:.4f}')
print(f'  Decision Tree (depth=4)  : AUC = {auc_tree:.4f}')


<a id='6'></a>
## 6. Blackbox: LIME on Random Forest

The blackbox explainers in InterpretML apply post-hoc explanation to any trained model.
We train a **Random Forest** — the most common model in clinical XAI literature — and
apply LIME via InterpretML's unified API, producing local explanations for individual patients.


In [ ]:
# ── 6.1 Train Random Forest ───────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf.fit(X_train, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'Random Forest AUC-ROC : {auc_rf:.4f}')


In [ ]:
# ── 6.2 LIME local explanations on RF ────────────────────────────────────────
lime_bb    = LimeTabular(rf, X_train, random_state=42)
lime_local = lime_bb.explain_local(
    X_test.iloc[:6], y_test.iloc[:6], name='LIME on RF'
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i in range(6):
    ld       = lime_local.data(i)
    pred     = rf.predict_proba(X_test.iloc[[i]])[0, 1]
    true_lbl = 'benign' if y_test.iloc[i] == 1 else 'malignant'
    pred_lbl = 'benign' if pred > 0.5 else 'malignant'
    top_l    = np.argsort(np.abs(ld['scores']))[-8:]
    cols     = ['#2ecc71' if ld['scores'][j] > 0 else '#e74c3c' for j in top_l]
    axes[i].barh(range(len(top_l)), [ld['scores'][j] for j in top_l], color=cols)
    axes[i].set_yticks(range(len(top_l)))
    axes[i].set_yticklabels([ld['names'][j][:30] for j in top_l], fontsize=7)
    axes[i].axvline(0, color='black', linewidth=0.6)
    axes[i].set_title(
        f'Patient {i} | True: {true_lbl}\nPred: {pred_lbl} (P={pred:.2f})',
        fontsize=8
    )
    axes[i].set_xlabel('LIME weight', fontsize=7)

g_p = mpatches.Patch(color='#2ecc71', label='Supports benign')
r_p = mpatches.Patch(color='#e74c3c', label='Supports malignant')
fig.legend(handles=[g_p, r_p], loc='lower center', ncol=2,
           fontsize=9, bbox_to_anchor=(0.5, -0.03))
fig.suptitle('LIME Blackbox Explanations on Random Forest (via InterpretML)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()


<a id='7'></a>
## 7. Blackbox: ShapKernel on Random Forest

`ShapKernel` applies model-agnostic KernelSHAP to any black-box model.
For tree-based models, the dedicated `shap.TreeExplainer` is faster and exact.
`ShapKernel` is most useful for models without a dedicated SHAP explainer
(e.g., SVMs, custom neural networks, federated models).


In [ ]:
# ── 7. ShapKernel on RF ───────────────────────────────────────────────────────
# ShapKernel(model, data, feature_names=...)
shap_bb = ShapKernel(
    rf,
    X_train.iloc[:100],
    feature_names = list(X_train.columns)
)
shap_local = shap_bb.explain_local(
    X_test.iloc[:5], y_test.iloc[:5], name='ShapKernel on RF'
)

# Aggregate mean |SHAP| across 5 patients
all_shap = []
for i in range(5):
    sd = shap_local.data(i)
    all_shap.append({n: abs(s) for n, s in zip(sd['names'], sd['scores'])})

shap_agg = pd.DataFrame(all_shap).mean().sort_values(ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(9, 5))
colours = plt.cm.Oranges(np.linspace(0.4, 0.9, len(shap_agg)))
ax.barh(range(len(shap_agg)), shap_agg.values, color=colours)
ax.set_yticks(range(len(shap_agg)))
ax.set_yticklabels(shap_agg.index, fontsize=9)
ax.set_xlabel('Mean |SHAP value| across 5 patients', fontsize=10)
ax.set_title('ShapKernel on Random Forest (via InterpretML)', fontsize=10, pad=10)
plt.tight_layout()
plt.show()


<a id='8'></a>
## 8. Blackbox: Partial Dependence Plot

`PartialDependence` computes the marginal effect of each feature on the model's predicted
probability, averaged across all patients in the training data. It answers:
*on average, how does the predicted probability change as this feature varies?*
This population-level view complements local explanations and is particularly informative
for identifying clinically relevant thresholds or non-linear effects.


In [ ]:
# ── 8. Partial Dependence Plot ────────────────────────────────────────────────
pdp        = PartialDependence(
    rf.predict_proba, X_train, feature_names=list(X_train.columns)
)
pdp_global = pdp.explain_global(name='PDP on RF')

top4_rf    = np.argsort(rf.feature_importances_)[-4:]

fig, axes  = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax_i, feat_i in enumerate(reversed(top4_rf)):
    pd_data = pdp_global.data(feat_i)
    # names = bin centres, scores = predicted probabilities (same length)
    x_vals  = np.array(pd_data['names'])
    y_vals  = np.array(pd_data['scores'])
    axes[ax_i].plot(x_vals, y_vals, color='#C0392B', linewidth=2,
                    marker='o', markersize=4)
    axes[ax_i].axhline(float(np.mean(y_vals)), color='gray', linewidth=0.8,
                        linestyle='--', alpha=0.7, label='Mean prediction')
    axes[ax_i].set_xlabel(feat_names[feat_i], fontsize=9)
    axes[ax_i].set_ylabel('P(benign) — population average', fontsize=8)
    axes[ax_i].set_title(f'PDP: {feat_names[feat_i]}', fontsize=9)
    axes[ax_i].legend(fontsize=7)
    axes[ax_i].spines['top'].set_visible(False)
    axes[ax_i].spines['right'].set_visible(False)

fig.suptitle('Partial Dependence Plots — Random Forest (via InterpretML)', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()


<a id='9'></a>
## 9. Blackbox: Morris Sensitivity Analysis

**Morris Sensitivity Analysis** estimates global feature importance through an
elementary-effects screening design — measuring how much the model's output changes
when feature values are perturbed systematically through the input space.
It is computationally efficient, scales to high-dimensional inputs, and requires
no assumptions about the model. Despite its utility, it appears in essentially no
papers in the 644-paper corpus reviewed in Chapter 4 — a significant underutilisation.


In [ ]:
# ── 9. Morris Sensitivity Analysis ───────────────────────────────────────────
morris        = MorrisSensitivity(
    rf.predict_proba,
    X_train.iloc[:100],
    feature_names = list(X_train.columns)
)
morris_global = morris.explain_global(name='Morris Sensitivity on RF')

md_data       = morris_global.data()
sorted_morris = np.argsort(md_data['scores'])[-15:]

fig, ax = plt.subplots(figsize=(9, 6))
colours = plt.cm.Reds(np.linspace(0.4, 0.9, len(sorted_morris)))
ax.barh(
    range(len(sorted_morris)),
    [md_data['scores'][i] for i in sorted_morris],
    color=colours
)
ax.set_yticks(range(len(sorted_morris)))
ax.set_yticklabels([md_data['names'][i] for i in sorted_morris], fontsize=9)
ax.set_xlabel('Mean absolute elementary effect (Morris mu*)', fontsize=10)
ax.set_title(
    'Morris Sensitivity Analysis — Random Forest (via InterpretML)\n'
    'Global, model-agnostic — efficient first-pass feature screening',
    fontsize=10, pad=10
)
plt.tight_layout()
plt.show()


<a id='10'></a>
## 10. Performance comparison

The EBM achieves near-equal AUC to the black-box Random Forest while requiring no post-hoc
explanation. This demonstrates that intrinsic interpretability does not require sacrificing
predictive accuracy on tabular clinical data — the central argument for EBM adoption in
medical AI.


In [ ]:
# ── 10. AUC comparison ───────────────────────────────────────────────────────
results = pd.DataFrame([
    {'Model': 'EBM (glassbox)',                   'AUC': auc_ebm,  'Type': 'Intrinsic'},
    {'Model': 'Logistic Regression (glassbox)',   'AUC': auc_lr,   'Type': 'Intrinsic'},
    {'Model': 'Decision Tree (glassbox)',          'AUC': auc_tree, 'Type': 'Intrinsic'},
    {'Model': 'Random Forest (blackbox)',          'AUC': auc_rf,   'Type': 'Post-hoc needed'},
])

type_cols = {'Intrinsic': '#2E75B6', 'Post-hoc needed': '#C0392B'}
bar_cols  = [type_cols[t] for t in results['Type']]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(results['Model'], results['AUC'], color=bar_cols, height=0.5)
ax.set_xlim(0.85, 1.005)
ax.set_xlabel('AUC-ROC', fontsize=10)
ax.set_title('Model Performance Comparison (Wisconsin Breast Cancer Dataset)', fontsize=10, pad=10)
for bar, val in zip(bars, results['AUC']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
b_p = mpatches.Patch(color='#2E75B6', label='Glassbox (intrinsically interpretable)')
r_p = mpatches.Patch(color='#C0392B', label='Blackbox (post-hoc explanation needed)')
ax.legend(handles=[b_p, r_p], fontsize=8.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(results.to_string(index=False))


<a id='11'></a>
## 11. Summary: when to use each method


In [ ]:
summary = pd.DataFrame([
    {'Method':'EBM','Category':'Glassbox',
     'Explanation':'Shape functions (intrinsic, exact)','Scope':'Global + Local',
     'Best for':'Tabular EHR; accuracy + interpretability required'},
    {'Method':'Logistic Regression','Category':'Glassbox',
     'Explanation':'Coefficients (intrinsic, exact)','Scope':'Global + Local',
     'Best for':'Regulatory audit; linear clinical risk scores'},
    {'Method':'Decision Tree','Category':'Glassbox',
     'Explanation':'Decision rules (intrinsic, exact)','Scope':'Global + Local',
     'Best for':'Clinician-readable rules; low-complexity settings'},
    {'Method':'LIME','Category':'Blackbox',
     'Explanation':'Local surrogate (post-hoc, approximate)','Scope':'Local',
     'Best for':'Any black-box model; imaging; text; case-by-case'},
    {'Method':'ShapKernel','Category':'Blackbox',
     'Explanation':'Shapley values (post-hoc, approximate)','Scope':'Local + Global',
     'Best for':'Models without a dedicated SHAP explainer'},
    {'Method':'Partial Dependence','Category':'Blackbox',
     'Explanation':'Marginal effect (post-hoc)','Scope':'Global',
     'Best for':'Population-level feature effect; threshold detection'},
    {'Method':'Morris Sensitivity','Category':'Blackbox',
     'Explanation':'Elementary effects (post-hoc)','Scope':'Global',
     'Best for':'Fast feature screening; high-dimensional inputs'},
])
print(summary.to_string(index=False))


---

## Key takeaways

1. **InterpretML is the only XAI library providing both glassbox and blackbox methods**
   under a single unified API.

2. **The EBM achieves blackbox-competitive AUC while remaining intrinsically interpretable.**
   Its shape functions ARE the model — not an approximation. Faithfulness is guaranteed
   by construction, not evaluated post-hoc.

3. **Glassbox models satisfy a stricter regulatory standard** than post-hoc methods.
   Because the explanation is the model, there is no approximation error to evaluate —
   directly addressing EU AI Act Article 13 and FDA SaMD transparency requirements.

4. **The blackbox wrappers allow InterpretML to explain any existing model**, making it
   practical when a model has already been trained and a post-hoc explanation is needed.

5. **Morris Sensitivity Analysis is underused in medical XAI** — appearing in essentially
   no papers in the 644-paper corpus of Chapter 4 — despite being a computationally
   efficient global feature screening method that scales to high-dimensional inputs.

---

## Further reading

- Nori, H., Jenkins, S., Koch, P. & Caruana, R. (2019). *InterpretML: A unified framework
  for machine learning interpretability.* arXiv:1909.09223.
- Lou, Y., Caruana, R., Gehrke, J. & Hooker, G. (2013). *Accurate intelligible models with
  pairwise interactions.* KDD 2013. — The EBM theoretical paper.
- InterpretML GitHub: https://github.com/interpretml/interpret
- InterpretML documentation: https://interpret.ml/docs
